# TOBIAS

In [1]:
import os, random, glob
from collections import defaultdict

def StartQsub(filename, nproc, memperproc, wd):
    f = open(filename, "w")
    f.write("#!/bin/bash\n\n")
    f.write("#$ -S /bin/bash\n")
    f.write(f"#$ -wd {wd}\n") 
    f.write("#$ -j y\n") # STDERR and STDOUT should be joined
    f.write(f"#$ -l mem_free={memperproc}G\n") 
    f.write("#$ -l scratch=25G\n") # job requires up to 2 GiB of local /scratch space
    f.write(f"#$ -pe smp {nproc}\n") #the job will be allotted n slots (“cores”) on a single machine
    f.write("#$ -l h_rt=48:00:00\n")
    f.write("#$ -l x86-64-v=4\n") #specifically for hint and tobias
    f.write("set -e\n")
    f.write("\n")

    f.write("module load CBI miniforge3\n")
    f.write("conda activate foot_progs\n")

    f.write("\n")
    f.write("start=$(date +%s)\n")
    return f


def _file_groups(directory, prefix):
    suffix = "_ftscore.bw"
    files = glob.glob(f"{directory}/**/{prefix}*{suffix}", recursive=True)

    grouped_files = defaultdict(list)
    for file in files:
        basename = os.path.basename(file)
        val = "_".join(basename.split('_')[1:2])  # Extract everything before the second underscore
        #val = val[1:]
        grouped_files[val].append(basename)
    return grouped_files
    
        
def WithinReplicates(cell_line, indir):
    maxvmem = 16 #GB
    nproc = 8
    memperproc = round(maxvmem/nproc)
    
    genome = "/pollard/data/projects/aseveritt/refdata-cellranger-arc-GRCh38-2020-A-2.0.0/fasta/genome.fa" 
    motif_file = "../05_footprinting/program_input_files/JASPAR2022_CORE_vertebrates_non-redundant_pfms_jaspar_nameFIX.txt"
    peakfile_path = f"../03_peakcalls/{cell_line}_filt_500bp.exclusion.bed.gz"
    
    grouped_files = _file_groups(indir, cell_line)
    
    for condition, includedfiles in grouped_files.items():
        qsub_script  = f"../10_DEbinding/tobias/qsubs/{cell_line}_{condition}.qsub"
        outdir = f"../10_DEbinding/tobias/within_rep/{cell_line}_{condition}/"
        includedfiles2 = [f"{indir}{i}" for i in includedfiles]
        
        f = StartQsub(qsub_script, nproc, memperproc, wd = f"../10_DEbinding/tobias/qsubs/")

        sge_ncpu = "${NSLOTS:-1}"
        command = f'''TOBIAS BINDetect \\
        --skip-excel \\
        --motifs {motif_file} \\
        --genome {genome} \\
        --peaks {peakfile_path} \\
        --cond_names rep1 rep2 rep3 \\
        --signals {" ".join(includedfiles2)} \\
        --outdir {outdir} \\
        --cores {sge_ncpu}\n\n'''
        f.write(command)
        
        command = f'''rm -r {outdir}*_MA*\n\n'''
        f.write(command)
        
        f.write('[[ -n "$JOB_ID" ]] && qstat -j "$JOB_ID"\n')
        f.write('echo "Elapsed Time: $(($end-$start)) seconds"\n')
        f.close()
    
    return





def AcrossReplicates(cell_line, indir, cond="whoopsies"):
    
    
    def _internal(replist, order, qsub_script, outdir):
        maxvmem = 16 #GB
        nproc = 8
        memperproc = round(maxvmem/nproc)
        
        genome = "/pollard/data/projects/aseveritt/refdata-cellranger-arc-GRCh38-2020-A-2.0.0/fasta/genome.fa" 
        motif_file = "../05_footprinting/program_input_files/JASPAR2022_CORE_vertebrates_non-redundant_pfms_jaspar_nameFIX.txt"
        peakfile_path = f"../03_peakcalls/{cell_line}_filt_500bp.exclusion.bed"
    
        f = StartQsub(qsub_script, nproc, memperproc, wd = f"../10_DEbinding/tobias/qsubs/")

        sge_ncpu = "${NSLOTS:-1}"
        command = f'''TOBIAS BINDetect \\
        --skip-excel \\
        --motifs {motif_file} \\
        --genome {genome} \\
        --peaks {peakfile_path} \\
        --cond_names {" ".join(map(str, theorder))} \\
        --signals {" ".join(replist)} \\
        --outdir {outdir} \\
        --cores {sge_ncpu}\n\n'''
        f.write(command)
        
        command = f'''rm -r {outdir}*_MA*\n\n'''
        f.write(command)
        
        f.write('[[ -n "$JOB_ID" ]] && qstat -j "$JOB_ID"\n')
        f.write('echo "Elapsed Time: $(($end-$start)) seconds"\n')
        f.close()
        return
        
        
    grouped_files = _file_groups(indir, cell_line)
    
    theorder = []; rep1=[]; rep2=[]; rep3=[]
    for key, value in sorted(grouped_files.items()): 
        theorder.append(key)
        rep1.append(f"{indir}{value[0]}")
        rep2.append(f"{indir}{value[1]}")
        rep3.append(f"{indir}{value[2]}")
        
    _internal(replist= rep1, 
              order = theorder, 
              qsub_script= f"../10_DEbinding/tobias/qsubs/{cell_line}_{cond}_rep1.qsub",
              outdir = f"../10_DEbinding/tobias/across_rep/{cell_line}_{cond}_rep1/")
#     _internal(replist= rep2, 
#               order = theorder, 
#               qsub_script= f"../10_DEbinding/tobias/qsubs/{cell_line}_{cond}_rep2.qsub",
#               outdir = f"../10_DEbinding/tobias/across_rep/{cell_line}_{cond}_rep2/")
#     _internal(replist= rep3, 
#               order = theorder, 
#               qsub_script= f"../10_DEbinding/tobias/qsubs/{cell_line}_{cond}_rep3.qsub",
#               outdir = f"../10_DEbinding/tobias/across_rep/{cell_line}_{cond}_rep3/")
    
    return





In [ ]:
Path(f"../10_DEbinding/tobias/qsubs/").mkdir(parents=True, exist_ok=True)
Path(f"../10_DEbinding/tobias/within_rep/").mkdir(parents=True, exist_ok=True)
Path(f"../10_DEbinding/tobias/across_rep/").mkdir(parents=True, exist_ok=True)

WithinReplicates("MCF7", "../05_footprinting/02_cells/tobias/")
WithinReplicates("HEPG2", "../05_footprinting/02_cells/tobias/")
WithinReplicates("K562", "../05_footprinting/02_cells/tobias/")

AcrossReplicates("MCF7", "../05_footprinting/02_cells/tobias/", "cells")
AcrossReplicates("HEPG2", "../05_footprinting/02_cells/tobias/", "cells")
AcrossReplicates("K562", "../05_footprinting/02_cells/tobias/", "cells")

In [137]:
WithinReplicates("MCF7", "../05_footprinting/03_reads/tobias/")
WithinReplicates("HEPG2", "../05_footprinting/03_reads/tobias/")
WithinReplicates("K562", "../05_footprinting/03_reads/tobias/")

AcrossReplicates("HEPG2", "../05_footprinting/03_reads/tobias/", "reads")
AcrossReplicates("MCF7", "../05_footprinting/03_reads/tobias/", "reads")
AcrossReplicates("K562", "../05_footprinting/03_reads/tobias/", "reads")

In [146]:
WithinReplicates("MCF7", "../05_footprinting/04_frip/tobias/")
WithinReplicates("HEPG2", "../05_footprinting/04_frip/tobias/")
WithinReplicates("K562", "../05_footprinting/04_frip/tobias/")

AcrossReplicates("MCF7", "../05_footprinting/04_frip/tobias/", "frip")
AcrossReplicates("HEPG2", "../05_footprinting/04_frip/tobias/", "frip")
AcrossReplicates("K562", "../05_footprinting/04_frip/tobias/", "frip")

In [2]:
### UNION FILE
replist=['../05_footprinting/01_original/union/tobias/HEPG2-cellFilt_corrected_ftscore.bw',
         '../05_footprinting/01_original/union/tobias/K562-cellFilt_corrected_ftscore.bw',
         '../05_footprinting/01_original/union/tobias/MCF7-cellFilt_corrected_ftscore.bw',
         '../05_footprinting/01_original/union/tobias/SKNSH-cellFilt_corrected_ftscore.bw']

namelist=['HEPG2', 'K562', 'MCF7', 'SKNSH']
qsub_script = "../10_DEbinding/tobias/qsubs/union.qsub"
outdir = "../10_DEbinding/tobias/union"

maxvmem = 16 #GB
nproc = 8
memperproc = round(maxvmem/nproc)

genome = "/pollard/data/projects/aseveritt/refdata-cellranger-arc-GRCh38-2020-A-2.0.0/fasta/genome.fa" 
motif_file = "../05_footprinting/program_input_files/JASPAR2022_CORE_vertebrates_non-redundant_pfms_jaspar_nameFIX.txt"
peakfile_path = f"../03_peakcalls/Union_filt_500bp.exclusion.bed.gz"


f = StartQsub(qsub_script, nproc, memperproc, wd = f"../10_DEbinding/tobias/qsubs/")

sge_ncpu = "${NSLOTS:-1}"
command = f'''TOBIAS BINDetect \\
--skip-excel \\
--motifs {motif_file} \\
--genome {genome} \\
--peaks {peakfile_path} \\
--cond_names {" ".join(namelist)} \\
--signals {" ".join(replist)} \\
--outdir {outdir} \\
--cores {sge_ncpu}\n\n'''
f.write(command)

command = f'''rm -r {outdir}*_MA*\n\n'''
f.write(command)

f.write('[[ -n "$JOB_ID" ]] && qstat -j "$JOB_ID"\n')
f.write('echo "Elapsed Time: $(($end-$start)) seconds"\n')
f.close()

# HINT

In [3]:
import os, random, glob
from collections import defaultdict

def StartQsub(filename, nproc, memperproc, wd):
    f = open(filename, "w")
    f.write("#!/bin/bash\n\n")
    f.write("#$ -S /bin/bash\n")
    f.write(f"#$ -wd {wd}\n") 
    f.write("#$ -j y\n") # STDERR and STDOUT should be joined
    f.write(f"#$ -l mem_free={memperproc}G\n") 
    f.write("#$ -l scratch=25G\n") # job requires up to 2 GiB of local /scratch space
    f.write(f"#$ -pe smp {nproc}\n") #the job will be allotted n slots (“cores”) on a single machine
    f.write("#$ -l h_rt=72:00:00\n")
    f.write("#$ -l x86-64-v=4\n") #specifically for hint and tobias
    f.write("set -e\n")
    f.write("\n")

    f.write("module load CBI miniforge3\n")
    f.write("conda activate foot_progs\n")

    f.write("\n")
    f.write("start=$(date +%s)\n")
    return f



def _file_groups(directory, prefix):
    suffix = "_mpbs.bed"
    files = glob.glob(f"{directory}/**/{prefix}*{suffix}", recursive=True)

    grouped_files = defaultdict(list)
    for file in files:
        basename = os.path.basename(file)
        val = "_".join(basename.split('_')[1:2])  # Extract everything before the second underscore
        #val = int(val[1:])
        grouped_files[val].append(basename)
    return grouped_files


        
def WithinReplicates(cell_line, indir, bamdir):
    
    def _internal(mpbs_lists, bam_lists, name_lists, qsub_script, outdir):
        maxvmem = 60 #GB
        nproc = 20
        memperproc = round(maxvmem/nproc)


        f = StartQsub(qsub_script, nproc, memperproc, 
                      wd = "../10_DEbinding/hint/qsubs/within_rep")

        sge_ncpu = "${NSLOTS:-1}"
        command = f'''rgt-hint differential \\
        --organism=hg38 \\
        --bc \\
        --no-lineplots \\
        --nc {sge_ncpu} \\
        --mpbs-files={",".join(mpbs_lists)} \\
        --reads-files={",".join(bam_lists)} \\
        --conditions={",".join(name_lists)} \\
        --output-location={localoutdir} \\
        --output-prefix={"_".join(name_lists)}\n\n'''
        #--output-profiles
        f.write(command)

        #command = f'''rm -r {outdir}*_MA*\n\n'''
        #f.write(command)

        f.write('[[ -n "$JOB_ID" ]] && qstat -j "$JOB_ID"\n')
        f.write('echo "Elapsed Time: $(($end-$start)) seconds"\n')
        f.close()
        return

    
    outdir = "../10_DEbinding/hint/"
    grouped_files = _file_groups(indir, cell_line)
    
    for condition, includedfiles in grouped_files.items():
        mpbs = [f"{indir}/{i}" for i in [includedfiles[0], includedfiles[1]] ]
        bams = [f"{bamdir}/{condition}/{i.replace('_mpbs.bed','.bam')}" for i in [includedfiles[0], includedfiles[1]] ]
        qsub_script  = f"{outdir}/qsubs/within_rep/{cell_line}_{condition}_rep1_rep2.qsub"
        localoutdir = f"{outdir}/within_rep/{cell_line}_{condition}/"
        _internal(mpbs, bams, ["rep1", "rep2"], qsub_script, localoutdir)

        mpbs = [f"{indir}/{i}" for i in [includedfiles[0], includedfiles[2]] ]
        bams = [f"{bamdir}/{condition}/{i.replace('_mpbs.bed','.bam')}" for i in [includedfiles[0], includedfiles[2]] ]
        qsub_script  = f"{outdir}/qsubs/within_rep/{cell_line}_{condition}_rep1_rep3.qsub"
        localoutdir = f"{outdir}/within_rep/{cell_line}_{condition}/"
        _internal(mpbs, bams, ["rep1", "rep3"], qsub_script, localoutdir)

        mpbs = [f"{indir}/{i}" for i in [includedfiles[1], includedfiles[2]] ]
        bams = [f"{bamdir}/{condition}/{i.replace('_mpbs.bed','.bam')}" for i in [includedfiles[1], includedfiles[2]] ]
        qsub_script  = f"{outdir}/qsubs/within_rep/{cell_line}_{condition}_rep2_rep3.qsub"
        localoutdir = f"{outdir}/within_rep/{cell_line}_{condition}/"
        _internal(mpbs, bams, ["rep2", "rep3"], qsub_script, localoutdir)
    
    
    return

def AcrossReplicates(cell_line, indir, bamdir, cond="whoopsies"):
    import itertools
    
    def _internal(mpbs_lists, bam_lists, name_lists, qsub_script, outdir):
        maxvmem = 60 #GB
        nproc = 20
        memperproc = round(maxvmem/nproc)

        f = StartQsub(qsub_script, nproc, memperproc, 
                      wd = "../10_DEbinding/hint/qsubs/across_rep")

        sge_ncpu = "${NSLOTS:-1}"
        command = f'''rgt-hint differential \\
        --organism=hg38 \\
        --bc \\
        --no-lineplots \\
        --nc {sge_ncpu} \\
        --mpbs-files={",".join(mpbs_lists)} \\
        --reads-files={",".join(bam_lists)} \\
        --conditions={",".join(name_lists)} \\
        --output-location={localoutdir} \\
        --output-prefix={"_".join(name_lists)}\n\n'''
        f.write(command)

        f.write('[[ -n "$JOB_ID" ]] && qstat -j "$JOB_ID"\n')
        f.write('echo "Elapsed Time: $(($end-$start)) seconds"\n')
        f.close()
        return

    
    outdir = "../10_DEbinding/hint/"    
    localoutdir = f"{outdir}/across_rep/{cell_line}_{cond}/"
    
    grouped_files = _file_groups(indir, cell_line)
    
    rep_list={}
    for key, value in sorted(grouped_files.items()): 
        rep_list[key] = f"{indir}{value[0]}"

    for i,j in itertools.combinations(rep_list.keys(), 2):
        mpbs = [rep_list[i], rep_list[j]]
        #print(mpbs)
        bams = [f"{bamdir}/{i}/{os.path.basename(rep_list[i]).replace('_mpbs.bed','.bam')}",
                f"{bamdir}/{j}/{os.path.basename(rep_list[j]).replace('_mpbs.bed','.bam')}"]
        qsub_script  = f"{outdir}/qsubs/across_rep/{cell_line}_{i}_{j}.qsub"
        _internal(mpbs, bams, [i, j], qsub_script, localoutdir)
    
    return

In [2]:
# Path(f"../10_DEbinding/hint/qsubs/").mkdir(parents=True, exist_ok=True)
# Path(f"../10_DEbinding/hint/within_rep/").mkdir(parents=True, exist_ok=True)

WithinReplicates(cell_line="MCF7", 
                 indir="../05_footprinting/02_cells/hint/",
                 bamdir="../04_downsampling/02_cells/")

WithinReplicates(cell_line="MCF7", 
                 indir="../05_footprinting/03_reads/hint/",
                 bamdir="../04_downsampling/03_reads/")

WithinReplicates(cell_line="MCF7", 
                 indir="../05_footprinting/04_frip/hint/",
                 bamdir="../04_downsampling/04_frip/")

In [3]:
WithinReplicates(cell_line="HEPG2", 
                 indir="../05_footprinting/02_cells/hint/",
                 bamdir="../04_downsampling/02_cells/")

WithinReplicates(cell_line="HEPG2", 
                 indir="../05_footprinting/03_reads/hint/",
                 bamdir="../04_downsampling/03_reads/")

WithinReplicates(cell_line="HEPG2", 
                 indir="../05_footprinting/04_frip/hint/",
                 bamdir="../04_downsampling/04_frip/")

In [4]:
WithinReplicates(cell_line="K562", 
                 indir="../05_footprinting/02_cells/hint/",
                 bamdir="../04_downsampling/02_cells/")

WithinReplicates(cell_line="K562", 
                 indir="../05_footprinting/03_reads/hint/",
                 bamdir="../04_downsampling/03_reads/")

WithinReplicates(cell_line="K562", 
                 indir="../05_footprinting/04_frip/hint/",
                 bamdir="../04_downsampling/04_frip/")

In [6]:
# Path(f"../10_DEbinding/hint/across_rep/").mkdir(parents=True, exist_ok=True)
# Path(f"../10_DEbinding/hint/qsubs/across_rep/").mkdir(parents=True, exist_ok=True)

AcrossReplicates(cell_line="MCF7", 
                 indir="../05_footprinting/02_cells/hint/",
                 bamdir="../04_downsampling/02_cells/", 
                 cond="cells")

AcrossReplicates(cell_line="HEPG2", 
                 indir="../05_footprinting/02_cells/hint/",
                 bamdir="../04_downsampling/02_cells/", 
                 cond="cells")

AcrossReplicates(cell_line="K562", 
                 indir="../05_footprinting/02_cells/hint/",
                 bamdir="../04_downsampling/02_cells/", 
                 cond="cells")

In [7]:
AcrossReplicates(cell_line="MCF7", 
                 indir="../05_footprinting/03_reads/hint/",
                 bamdir="../04_downsampling/03_reads/", 
                 cond="reads")

AcrossReplicates(cell_line="HEPG2", 
                 indir="../05_footprinting/03_reads/hint/",
                 bamdir="../04_downsampling/03_reads/", 
                 cond="reads")

AcrossReplicates(cell_line="K562", 
                 indir="../05_footprinting/03_reads/hint/",
                 bamdir="../04_downsampling/03_reads/", 
                 cond="reads")

In [8]:
AcrossReplicates(cell_line="MCF7", 
                 indir="../05_footprinting/04_frip/hint/",
                 bamdir="../04_downsampling/04_frip/", 
                 cond="frip")

AcrossReplicates(cell_line="HEPG2", 
                 indir="../05_footprinting/04_frip/hint/",
                 bamdir="../04_downsampling/04_frip/", 
                 cond="frip")

AcrossReplicates(cell_line="K562", 
                 indir="../05_footprinting/04_frip/hint/",
                 bamdir="../04_downsampling/04_frip/", 
                 cond="frip")

In [9]:
###ORIGINAL 

import itertools

def _internal(mpbs_lists, bam_lists, name_lists, qsub_script, outdir):
    maxvmem = 60 #GB
    nproc = 20
    memperproc = round(maxvmem/nproc)


    f = StartQsub(qsub_script, nproc, memperproc, 
                  wd = "../10_DEbinding/hint/qsubs/union")

    sge_ncpu = "${NSLOTS:-1}"
    command = f'''rgt-hint differential \\
    --organism=hg38 \\
    --bc \\
    --no-lineplots \\
    --nc {sge_ncpu} \\
    --mpbs-files={",".join(mpbs_lists)} \\
    --reads-files={",".join(bam_lists)} \\
    --conditions={",".join(name_lists)} \\
    --output-location={localoutdir} \\
    --output-prefix={"_".join(name_lists)}\n\n'''
    f.write(command)

    f.write('[[ -n "$JOB_ID" ]] && qstat -j "$JOB_ID"\n')
    f.write('echo "Elapsed Time: $(($end-$start)) seconds"\n')
    f.close()
    return


outdir = "../10_DEbinding/hint/"    
localoutdir = f"{outdir}/union/"

mpbs_dir = "../05_footprinting/01_original/union/hint/"
bam_dir = "../03_filtered_bams/"

for i,j in itertools.combinations(["HEPG2", "K562", "MCF7", "SKNSH"], 2):
    mpbs = [f"{mpbs_dir}/{i}-cellFilt_mpbs.bed", f"{mpbs_dir}/{j}-cellFilt_mpbs.bed"]
    bams = [f"{bam_dir}/{i}-cellFilt.bam", f"{bam_dir}/{j}-cellFilt.bam"]
    qsub_script  = f"{outdir}/qsubs/union/union_{i}_{j}.qsub"
    
    _internal(mpbs, bams, [i, j], qsub_script, localoutdir)